In [1]:
import numpy as np
import pandas as pd

TRAIN = 'data/train.csv'
TEST = 'data/test.csv'

all_data = pd.read_csv(TRAIN)
test = pd.read_csv(TEST)

print(f'Train shape: {all_data.shape}')
print(all_data.head())

Train shape: (7613, 5)
   id keyword location                                               text  \
0   1     NaN      NaN  Our Deeds are the Reason of this #earthquake M...   
1   4     NaN      NaN             Forest fire near La Ronge Sask. Canada   
2   5     NaN      NaN  All residents asked to 'shelter in place' are ...   
3   6     NaN      NaN  13,000 people receive #wildfires evacuation or...   
4   7     NaN      NaN  Just got sent this photo from Ruby #Alaska as ...   

   target  
0       1  
1       1  
2       1  
3       1  
4       1  


In [2]:
print(all_data.isnull().sum())
print(test.isnull().sum())

id             0
keyword       61
location    2533
text           0
target         0
dtype: int64
id             0
keyword       26
location    1105
text           0
dtype: int64


In [3]:
print(all_data['keyword'].value_counts())
print(all_data['location'].value_counts())

keyword
fatalities               45
deluge                   42
armageddon               42
damage                   41
body%20bags              41
                         ..
forest%20fire            19
epicentre                12
threat                   11
inundation               10
radiation%20emergency     9
Name: count, Length: 221, dtype: int64
location
USA                            104
New York                        71
United States                   50
London                          45
Canada                          29
                              ... 
Click the link below, okay       1
Milwaukee County                 1
Gwersyllt, Wales                 1
Primum non nocere                1
Alabama, USA                     1
Name: count, Length: 3341, dtype: int64


In [4]:
# all_data['location'].fillna('Unknown', inplace=True)
# all_data['keyword'].fillna('Unknown', inplace=True)
# test['location'].fillna('Unknown', inplace=True)
# test['keyword'].fillna('Unknown', inplace=True)

import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

all_data['text'] = [str(str(all_data['keyword'][i]).replace("nan", "") + "\n" + all_data['text'][i]) for i in range(len(all_data))]
test['text'] = [str(str(test['keyword'][i]).replace("nan", "") + "\n" + test['text'][i]) for i in range(len(test))]

all_data['text'] = all_data['text'].apply(clean_text)
test['text'] = test['text'].apply(clean_text)

print(all_data['text'][:10])

0     our deeds are the reason of this earthquake m...
1                forest fire near la ronge sask canada
2     all residents asked to shelter in place are b...
3     13000 people receive wildfires evacuation ord...
4     just got sent this photo from ruby alaska as ...
5     rockyfire update california hwy 20 closed in ...
6     flood disaster heavy rain causes flash floodi...
7     im on top of the hill and i can see a fire in...
8     theres an emergency evacuation happening now ...
9     im afraid that the tornado is coming to our area
Name: text, dtype: object


In [5]:
drop_cols = ['id', 'keyword', 'location']
# drop_cols = ['id', 'location']
all_data.drop(columns=drop_cols, inplace=True)
test_ids = test['id']
test.drop(columns=drop_cols, inplace=True)

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score

train, val = train_test_split(all_data, test_size=0.2, random_state=42)
X_train = train['text']
y_train = train['target']
X_val = val['text']
y_val = val['target']
X_test = test['text']

MAX_FEATURES = 10000

vectorizer = TfidfVectorizer(max_features=MAX_FEATURES)

X_train = vectorizer.fit_transform(X_train).toarray()
X_val = vectorizer.transform(X_val).toarray()
X_test = vectorizer.transform(X_test).toarray()

In [7]:
print(X_train)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [8]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(random_state=42, n_jobs=-1)

model_rf.fit(X_train, y_train)

rf_preds = model_rf.predict(X_val)

f1 = f1_score(y_val, rf_preds)
print(f'Validation F1 Score: {f1}')

Validation F1 Score: 0.7130434782608696


In [9]:
X_train = all_data['text']
y_train = all_data['target']
X_test = test['text']

MAX_FEATURES = 10000

vectorizer = TfidfVectorizer(max_features=MAX_FEATURES)

X_train = vectorizer.fit_transform(X_train).toarray()
X_test = vectorizer.transform(X_test).toarray()

model_rf = RandomForestClassifier(random_state=42, n_jobs=-1)

model_rf.fit(X_train, y_train)

preds = model_rf.predict(X_test)

In [10]:
submission = pd.DataFrame({
    'id': test_ids, 
    'target': preds
})

submission.to_csv('submissions/tfidf_rf.csv', index=False)